<a href="https://colab.research.google.com/github/Mariodalmo/BigDive/blob/master/notebooks/claude_asks_gemini_writes_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Claude Asks NotebookLM → Gemini Synthesizes → Claude Writes Code

This notebook demonstrates a **three-step AI pipeline** where two large language models collaborate to produce correct, domain-grounded Python code:

```
Your Task
   │
   ▼
┌─────────────────────────────────┐
│  Step 1 · Claude (Anthropic)    │  ← Formulates a precise domain question
│  Model: claude-opus-4-6         │
│  Thinking: adaptive             │
└─────────────┬───────────────────┘
              │ domain question
              ▼
┌─────────────────────────────────┐
│  Step 2 · Gemini (Google)       │  ← Synthesizes authoritative domain knowledge
│  Model: gemini-2.0-flash        │    (acts as NotebookLM / domain expert)
└─────────────┬───────────────────┘
              │ domain knowledge
              ▼
┌─────────────────────────────────┐
│  Step 3 · Claude (Anthropic)    │  ← Writes correct code grounded in that knowledge
│  Model: claude-opus-4-6         │
│  Thinking: adaptive             │
└─────────────────────────────────┘
              │
              ▼
        ✅ Correct Code
```

## Why this works

Claude excels at reasoning, code generation, and knowing *what* to ask. Gemini has deep knowledge of Google's own models (like MedGemma) and their APIs. By having Claude **delegate the domain research** to Gemini, the final code is grounded in authoritative, up-to-date implementation details — reducing hallucinations about API signatures, parameters, and best practices.

## Example use case

This notebook demonstrates the pipeline for the task:
> *"Write Python code to load MedGemma-4B and analyse a chest X-ray image, returning a structured radiology report."*

## Setup

You will need two API keys:

| Key | Where to get it |
|-----|-----------------|
| `ANTHROPIC_API_KEY` | [console.anthropic.com](https://console.anthropic.com/) |
| `GOOGLE_API_KEY` | [aistudio.google.com](https://aistudio.google.com/app/apikey) |

If running in **Google Colab**, store them in the Secrets panel (🔑) with the names above.

In [ ]:
! pip install --upgrade --quiet anthropic google-generativeai

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
else:
    # When running locally, set these in your shell before launching Jupyter:
    #   export ANTHROPIC_API_KEY="sk-ant-..."
    #   export GOOGLE_API_KEY="AIza..."
    pass

assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY is not set"
assert os.environ.get("GOOGLE_API_KEY"), "GOOGLE_API_KEY is not set"

print("✅ API keys loaded")

## Initialise clients

In [ ]:
import anthropic
import google.generativeai as genai
from IPython.display import Markdown, display

# Claude client (Anthropic)
claude = anthropic.Anthropic()
CLAUDE_MODEL = "claude-opus-4-6"  # Most capable Claude model

# Gemini client (Google) — acts as the NotebookLM / domain-knowledge layer
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
gemini = genai.GenerativeModel("gemini-2.0-flash")

print(f"Claude model : {CLAUDE_MODEL}")
print(f"Gemini model : gemini-2.0-flash")

## Pipeline implementation

The `claude_asks_gemini_writes` function encapsulates the full three-step pipeline.

Both Claude calls use **adaptive thinking** (`thinking: {"type": "adaptive"}`), which lets Claude decide when and how much internal reasoning to apply — ideal for complex tasks like formulating precise questions and generating correct code.

In [ ]:
def _extract_text(response: anthropic.types.Message) -> str:
    """Extract the first text block from a Claude response."""
    for block in response.content:
        if block.type == "text":
            return block.text
    return ""


def claude_asks_gemini_writes(task: str, verbose: bool = True) -> dict:
    """
    Three-step pipeline: Claude asks → Gemini synthesizes → Claude writes code.

    Args:
        task:    Plain-English description of the code to produce.
        verbose: Print each step's output as Markdown.

    Returns:
        A dict with keys:
            'question'         – the domain question Claude sent to Gemini
            'domain_knowledge' – Gemini's authoritative answer
            'code'             – the final Python code Claude produced
    """

    # ──────────────────────────────────────────────────────────────────────────
    # STEP 1 · Claude formulates a precise domain question for Gemini
    # ──────────────────────────────────────────────────────────────────────────
    if verbose:
        display(Markdown("---\n### Step 1 · Claude formulates a domain question"))

    step1 = claude.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=512,
        thinking={"type": "adaptive"},
        messages=[
            {
                "role": "user",
                "content": (
                    f"You are about to write Python code for the following task:\n\n"
                    f"TASK: {task}\n\n"
                    "Before writing any code, you must consult Gemini to get authoritative "
                    "domain knowledge. Formulate ONE focused technical question whose answer "
                    "will give you everything needed to write correct, production-ready code.\n\n"
                    "Output ONLY the question — no preamble, no explanation."
                ),
            }
        ],
    )
    question = _extract_text(step1).strip()

    if verbose:
        display(Markdown(f"**Claude's question to Gemini:**\n\n> {question}"))

    # ──────────────────────────────────────────────────────────────────────────
    # STEP 2 · Gemini synthesizes authoritative domain knowledge
    # ──────────────────────────────────────────────────────────────────────────
    if verbose:
        display(Markdown("---\n### Step 2 · Gemini synthesizes domain knowledge"))

    gemini_response = gemini.generate_content(question)
    domain_knowledge = gemini_response.text

    if verbose:
        display(Markdown(f"**Gemini's answer:**\n\n{domain_knowledge}"))

    # ──────────────────────────────────────────────────────────────────────────
    # STEP 3 · Claude writes correct code, grounded in Gemini's answer
    # ──────────────────────────────────────────────────────────────────────────
    if verbose:
        display(Markdown("---\n### Step 3 · Claude writes correct code"))

    step3 = claude.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=4096,
        thinking={"type": "adaptive"},
        messages=[
            {
                "role": "user",
                "content": (
                    f"Write Python code to accomplish the following task:\n\n"
                    f"TASK: {task}\n\n"
                    "Use the domain knowledge below — provided by Gemini — to ensure every "
                    "API call, parameter name, and model identifier is correct.\n\n"
                    f"DOMAIN KNOWLEDGE FROM GEMINI:\n{domain_knowledge}\n\n"
                    "Requirements:\n"
                    "- Include all necessary imports\n"
                    "- Add concise inline comments explaining non-obvious choices\n"
                    "- Handle common error cases gracefully\n"
                    "- Make the code runnable in Google Colab\n\n"
                    "Output ONLY the Python code — no markdown fences, no prose."
                ),
            }
        ],
    )
    code = _extract_text(step3).strip()

    if verbose:
        display(Markdown(f"---\n### ✅ Generated code\n\n```python\n{code}\n```"))

    return {
        "question": question,
        "domain_knowledge": domain_knowledge,
        "code": code,
    }

## Run the pipeline

We ask the pipeline to generate code for analysing a chest X-ray with **MedGemma-4B** — a Google model that Gemini knows authoritatively (API names, processor arguments, generation kwargs, etc.).

Modify `TASK` below to try your own coding challenge.

In [ ]:
TASK = (
    "Load google/medgemma-4b-it from Hugging Face using the Transformers library, "
    "download a sample chest X-ray image, run inference with a radiologist system prompt, "
    "and print a structured radiology report. "
    "The code must run on a T4 GPU in Google Colab."
)

result = claude_asks_gemini_writes(TASK, verbose=True)

## Save the generated code to a file

The `result` dict gives programmatic access to all three pipeline artefacts.

In [ ]:
output_path = "generated_medgemma_demo.py"

with open(output_path, "w") as f:
    f.write(result["code"])

print(f"Code written to {output_path}")
print(f"\nPipeline artefacts available in `result`:")
print(f"  result['question']          → {len(result['question'])} chars")
print(f"  result['domain_knowledge']  → {len(result['domain_knowledge'])} chars")
print(f"  result['code']              → {len(result['code'])} chars")

## Try a different task

The pipeline is task-agnostic. Here are some ideas:

```python
# MedGemma 27B with thinking mode
result = claude_asks_gemini_writes(
    "Load google/medgemma-27b-it with 4-bit quantization and use thinking mode "
    "to answer a complex pathology question about diabetic retinopathy."
)

# Batch inference
result = claude_asks_gemini_writes(
    "Write code to run MedGemma-4B inference on a directory of DICOM images "
    "and save the reports as a CSV file."
)

# Fine-tuning setup
result = claude_asks_gemini_writes(
    "Set up a LoRA fine-tuning loop for MedGemma-4B on a custom clinical QA dataset "
    "using PEFT and the Hugging Face Trainer API."
)
```

## Next steps

- Swap `gemini-2.0-flash` for **NotebookLM's underlying API** once Google exposes it publicly — the pipeline interface stays identical.
- Add a **verification step** where Claude reads the generated code and checks it against the domain knowledge before returning it.
- Extend the pipeline to generate **unit tests** alongside the code.
- See the [MedGemma quick-start notebook](quick_start_with_hugging_face.ipynb) to run the generated code on actual medical images.